In [0]:
silver_df = df.dropDuplicates(["SaleID"])

In [0]:
df = spark.sql("""
    SELECT *
    FROM mythri_databricks.sales_bronze.raw_sales
""")

df.show()

In [0]:
silver_df = df.dropDuplicates(["SaleID"])

In [0]:
silver_df.show()

In [0]:
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "mythri_databricks.sales_silver.cleaned_sales"
    )

In [0]:
%sql
SELECT *
FROM mythri_databricks.sales_silver.cleaned_sales;

In [0]:
silver_df = spark.sql("""
    SELECT *
    FROM mythri_databricks.sales_silver.cleaned_sales
""")

silver_df.show()

In [0]:
from pyspark.sql import functions as F

region_gold = (
    silver_df
    .groupBy("Region")
    .agg(
        F.countDistinct("SaleID").alias("SaleCount"),
        F.sum("SalesAmount").alias("TotalSales"),
        F.avg("SalesAmount").alias("AverageSale"),
        F.max("SalesAmount").alias("HighestSale"),
        F.min("SalesAmount").alias("LowestSale")
    )
    .orderBy(
        F.col("TotalSales").desc()
    )
)